In [ ]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import chroma_db
from typing import List, Dict, Tuple, Optional
import uuid

print("Day 13 - BM25 + Hybrid Search")

embedder =SentenceTransformer("all-MiniLM-L6-v2")
print("ready")

Day 13 - BM25 + Hybrid Search
ready


In [2]:
print("=== BM25 - How It Works ===\n")

# BM25 is a keyword search algorithm
# It scores documents based on:
# 1. Term frequency - how often the query word appear in the doc?
# 2. Inverse docuement frequency - how rare is the word across all docs?
# 3. Document length normalization - penalizes very long documents

# Simple example 
corpus = [
    "hybrid search combines BM25 and vector search",
    "BM25 is a keyword search algorithm for retrieval",
    "vector search uses embeddings for semantic similarity",
    "RAG combines retrieval and generation for better answers",
    "BM25 ranks documents based on keyword frequency and rarity"
]

# Tokenize - BM25 works on word lists not raw strings
tokenized_corpus = [doc.lower().split() for doc in corpus]

print("Tokenized corpus:")
for i ,tokens in enumerate(tokenized_corpus):
    print(f" Doc {i}: {tokens}")

# Build BM25 index
bm25 = BM25Okapi(tokenized_corpus)
print(f"\nBM25 index built on {len(corpus)} documents")

# Query 1 - keyword query 
query1 = "BM25 keyword search"
tokenized_query1 = query1.lower().split()
scores1 = bm25.get_scores(tokenized_query1)

print(f"\nQuery: '{query1}'")
print("BM25 Scores:")
for i, score in enumerate(scores1):
    print(f"    Doc {i}: {score:.4f} - {corpus[i][:50]}...")

# Query 2 - exact product code (where vector search fails)
query2 = "BM25"
tokenized_query2 = query2.lower().split()
scores2 = bm25.get_scores(tokenized_query2)

print(f"Query: '{query2}'")
print("BM25 scores: ")
ranked = sorted(enumerate(scores2), key= lambda x: x[1], reverse=True)
for idx, score in ranked:
    print(f" Doc {idx}: {score:.4f} - {corpus[idx][:50]}...")

=== BM25 - How It Works ===

Tokenized corpus:
 Doc 0: ['hybrid', 'search', 'combines', 'bm25', 'and', 'vector', 'search']
 Doc 1: ['bm25', 'is', 'a', 'keyword', 'search', 'algorithm', 'for', 'retrieval']
 Doc 2: ['vector', 'search', 'uses', 'embeddings', 'for', 'semantic', 'similarity']
 Doc 3: ['rag', 'combines', 'retrieval', 'and', 'generation', 'for', 'better', 'answers']
 Doc 4: ['bm25', 'ranks', 'documents', 'based', 'on', 'keyword', 'frequency', 'and', 'rarity']

BM25 index built on 5 documents

Query: 'BM25 keyword search'
BM25 Scores:
    Doc 0: 0.4802 - hybrid search combines BM25 and vector search...
    Doc 1: 0.7086 - BM25 is a keyword search algorithm for retrieval...
    Doc 2: 0.1993 - vector search uses embeddings for semantic similar...
    Doc 3: 0.0000 - RAG combines retrieval and generation for better a...
    Doc 4: 0.4925 - BM25 ranks documents based on keyword frequency an...
Query: 'BM25'
BM25 scores: 
 Doc 0: 0.1993 - hybrid search combines BM25 and vector sea

In [5]:
print("=== Reciprocal Rank Fusion (RRF) ===\n")

def reciprocal_rank_fusion(
        result_sets: List[List[Tuple[str, float]]],
        k: int =60
)-> List[Tuple[str, float]]:
    """
    Merge multiple ranked result sets using RRF.

    Args:
        result_set: list of [(doc_id, score)] lists
                    each list is a ranked result from one search method
        k: constant that controls rank influence (default 60)

    Returns:
        Merged and re-ranked list of (doc_id, rrf_score)
    """
    rrf_scores= {}

    for result_set in result_sets:
        for rank, (doc_id, score) in enumerate(result_set):
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = 0.0
            # RRF formula: 1/(k+rank)
            # Higher rank (lower number) = higher score
            rrf_scores[doc_id] += 1.0/(k+ rank+1)

    # sort by RRF score descending 
    merged = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return merged
# Simulate two result sets
# Vector search results
vector_results = [
    ("doc_A", 0.92), # rank 1
    ("doc_B", 0.85), # rank 2
    ("doc_C", 0.78), # rank 3
    ("doc_D", 0.71)  # rank 4
]

# BM25 results - different ranking
bm25_results = [
    ("doc_C", 8.5), # rank 1 - doc_C ranks higher in BM25
    ("doc_E", 7.2), # rank 2 - new doc not in vector results
    ("doc_A", 6.1), # rank 3
    ("doc_B", 4.8)  # rank 4
]

print("Vector search ranking:")
for rank, (doc_id, score) in enumerate(vector_results):
    print(f"  Rank {rank+1}: {doc_id} (Score: {score:.6f})")

print("\nBM25 ranking:")
for rank, (doc_id, score) in enumerate(bm25_results):
    print(f" Rank {rank+1}: {doc_id} (score: {score})")

# Merge with RRf
merged = reciprocal_rank_fusion([vector_results, bm25_results])

print("\nRRF merged ranking:")
for rank, (doc_id, rrf_score) in enumerate(merged):
    print(f"  Rank {rank+1}: {doc_id}  (rrf_score: {rrf_score:.6f})")

print("\nNotice:")
print("  doc_C moved up - ranked 1st in BM25, 3rd in vector -> merged higher")
print(" doc_E appered - only in BM25 but still included in final results")
print(" doc_A stayed high  - ranked 1st in vector, 3rd in BM25")

=== Reciprocal Rank Fusion (RRF) ===

Vector search ranking:
  Rank 1: doc_A (Score: 0.920000)
  Rank 2: doc_B (Score: 0.850000)
  Rank 3: doc_C (Score: 0.780000)
  Rank 4: doc_D (Score: 0.710000)

BM25 ranking:
 Rank 1: doc_C (score: 8.5)
 Rank 2: doc_E (score: 7.2)
 Rank 3: doc_A (score: 6.1)
 Rank 4: doc_B (score: 4.8)

RRF merged ranking:
  Rank 1: doc_A  (rrf_score: 0.032266)
  Rank 2: doc_C  (rrf_score: 0.032266)
  Rank 3: doc_B  (rrf_score: 0.031754)
  Rank 4: doc_E  (rrf_score: 0.016129)
  Rank 5: doc_D  (rrf_score: 0.015625)

Notice:
  doc_C moved up - ranked 1st in BM25, 3rd in vector -> merged higher
 doc_E appered - only in BM25 but still included in final results
 doc_A stayed high  - ranked 1st in vector, 3rd in BM25


In [13]:
from typing import List, Dict, Optional, Tuple
import uuid
import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

class HybridSearchRetriever:
    """
    Combines BM25 keyword search with ChromaDB vector search.
    Uses RRF to merge results.
    This is the core retriever for your Enterprise RAG pipeline.
    """
    
    def __init__(self, org_id: str, persist_path: str = "./chroma_hybrid"):
        self.org_id = org_id
        self.embedder = SentenceTransformer("all-MiniLM-L6-v2")
        
        # ChromaDB for vector search
        self.chroma_client = chromadb.PersistentClient(path=persist_path)
        self.collection = self.chroma_client.get_or_create_collection(
            name=f"org_{org_id}"
        )
        
        # BM25 for keyword search
        self.bm25 = None
        self.bm25_corpus = []      # raw texts
        self.bm25_ids = []         # document IDs parallel to corpus
        self.bm25_metadatas = []   # metadata parallel to corpus
    
    def add_documents(
        self,
        texts: List[str],
        metadatas: List[Dict],
        ids: Optional[List[str]] = None
    ) -> None:
        if ids is None:
            ids = [str(uuid.uuid4())[:8] for _ in texts]
        
        # Add to ChromaDB
        embeddings = self.embedder.encode(texts).tolist()
        self.collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )
        
        # Add to BM25 index
        self.bm25_corpus.extend(texts)
        self.bm25_ids.extend(ids)
        self.bm25_metadatas.extend(metadatas)
        
        # Rebuild BM25 index
        tokenized = [doc.lower().split() for doc in self.bm25_corpus]
        self.bm25 = BM25Okapi(tokenized)
        
        print(f"[{self.org_id}] Added {len(texts)} docs | Total: {len(self.bm25_corpus)}")
    
    def _vector_search(self, query: str, n: int) -> List[Tuple[str, float]]:
        """Vector search using ChromaDB"""
        query_embedding = self.embedder.encode(query).tolist()
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=min(n, self.collection.count())
        )
        return [
            (results['ids'][0][i], 1 - results['distances'][0][i])
            for i in range(len(results['ids'][0]))
        ]
    
    def _bm25_search(self, query: str, n: int) -> List[Tuple[str, float]]:
        """BM25 keyword search"""
        if self.bm25 is None:
            return []
        tokenized_query = query.lower().split()
        scores = self.bm25.get_scores(tokenized_query)
        ranked = sorted(
            enumerate(scores),
            key=lambda x: x[1],
            reverse=True
        )[:n]
        return [(self.bm25_ids[idx], score) for idx, score in ranked]
    
    def search(
        self,
        query: str,
        n_results: int = 3,
        search_type: str = "hybrid"  # "hybrid", "vector", "bm25"
    ) -> List[Dict]:
        """
        Search using specified strategy.
        hybrid = BM25 + vector merged with RRF
        """
        if search_type == "vector":
            results = self._vector_search(query, n_results)
        elif search_type == "bm25":
            results = self._bm25_search(query, n_results)
        else:  # hybrid
            vector_results = self._vector_search(query, n_results * 2)
            bm25_results = self._bm25_search(query, n_results * 2)
            results = reciprocal_rank_fusion([vector_results, bm25_results])
            results = results[:n_results]
        
        # Format results with full document info
        formatted = []
        for doc_id, score in results:
            # Get text and metadata from BM25 corpus
            if doc_id in self.bm25_ids:
                idx = self.bm25_ids.index(doc_id)
                formatted.append({
                    "id": doc_id,
                    "text": self.bm25_corpus[idx],
                    "score": score,
                    "metadata": self.bm25_metadatas[idx],
                    "search_type": search_type
                })
        
        return formatted
    
    def benchmark(self, query: str, n_results: int = 3) -> None:
        """Compare all 3 search strategies side by side"""
        print(f"Query: '{query}'\n")
        
        for strategy in ["vector", "bm25", "hybrid"]:
            results = self.search(query, n_results, strategy)
            print(f"=== {strategy.upper()} ===")
            for i, r in enumerate(results):
                print(f"  Rank {i+1}: [{r['score']:.4f}] {r['text'][:60]}...")
            print()


# Test with documents that highlight BM25 vs vector differences
print("=== HybridSearchRetriever Test ===\n")

import uuid
from typing import Optional

retriever = HybridSearchRetriever("test_org", "./chroma_hybrid")

# Add documents
retriever.add_documents(
    texts=[
        "RAG-2024-ENTERPRISE is the product code for our RAG system",
        "Hybrid search combines BM25 and vector search for retrieval",
        "The enterprise RAG pipeline uses ChromaDB for vector storage",
        "BM25 algorithm ranks documents based on keyword frequency",
        "Vector embeddings capture semantic meaning of text",
        "RAG-2024-ENTERPRISE supports multi-tenant document isolation",
        "FastAPI backend handles authentication and request routing",
        "Evaluation metrics include faithfulness and answer relevancy"
    ],
    metadatas=[
        {"source": "products.pdf", "topic": "product"},
        {"source": "architecture.pdf", "topic": "retrieval"},
        {"source": "architecture.pdf", "topic": "database"},
        {"source": "algorithms.pdf", "topic": "retrieval"},
        {"source": "ml.pdf", "topic": "embeddings"},
        {"source": "products.pdf", "topic": "product"},
        {"source": "deployment.pdf", "topic": "backend"},
        {"source": "evaluation.pdf", "topic": "metrics"}
    ]
)

# Benchmark queries
print("\n" + "="*60)
retriever.benchmark("RAG-2024-ENTERPRISE product information")

print("="*60)
retriever.benchmark("how does semantic search work")

=== HybridSearchRetriever Test ===

[test_org] Added 8 docs | Total: 8

Query: 'RAG-2024-ENTERPRISE product information'

=== VECTOR ===
  Rank 1: [0.5225] RAG-2024-ENTERPRISE is the product code for our RAG system...
  Rank 2: [-0.3004] The enterprise RAG pipeline uses ChromaDB for vector storage...
  Rank 3: [-0.3426] RAG-2024-ENTERPRISE supports multi-tenant document isolation...

=== BM25 ===
  Rank 1: [2.3724] RAG-2024-ENTERPRISE is the product code for our RAG system...
  Rank 2: [1.1307] RAG-2024-ENTERPRISE supports multi-tenant document isolation...
  Rank 3: [0.0000] Hybrid search combines BM25 and vector search for retrieval...

=== HYBRID ===
  Rank 1: [0.0328] RAG-2024-ENTERPRISE is the product code for our RAG system...
  Rank 2: [0.0320] RAG-2024-ENTERPRISE supports multi-tenant document isolation...
  Rank 3: [0.0318] The enterprise RAG pipeline uses ChromaDB for vector storage...

Query: 'how does semantic search work'

=== VECTOR ===
  Rank 1: [0.0078] Hybrid search co